# Merge function in pandas

### Import necessary libraries

In [6]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

## Merge multiple dataFrames into a single dataFrame

### Create data using python dictionaries

In [19]:
# create a dummy data for INDIA cities
india_weather_humidity = {
    "city": ["mumbai"] * 5 + ["lucknow"] * 5 + ["delhi"] * 5,
    "humidity": [
        # Mumbai (humid, coastal)
        82, 85, 80, 88, 90,
        # Lucknow (moderate humidity)
        60, 65, 70, 55, 62,
        # Delhi (dry, lower humidity)
        45, 50, 40, 42, 47
    ]
}

india_weather_precipitation = {
    "city": ["mumbai"] * 5 + ["lucknow"] * 5 + ["delhi"] * 5,
    "precipitation": [
        # Mumbai (frequent rain in monsoon season)
        15.2, 12.5, 20.1, 10.0, 18.3,
        # Lucknow (occasional rain)
        5.0, 2.2, 0.0, 3.1, 1.5,
        # Delhi (less rainfall generally)
        0.0, 0.5, 0.0, 1.2, 0.0
    ]
}
india_weather_temperature = {
    "city": ["mumbai"] * 5 + ["lucknow"] * 5 + ["delhi"] * 5,
    "Temperature": [
        # Mumbai (warm and stable)
        29, 30, 31, 28, 29,
        # Lucknow (hotter variation)
        34, 36, 33, 35, 37,
        # Delhi (very hot summers)
        38, 40, 39, 41, 42
    ]
}

### Create dataFrames from the python dictionaries created earlier

In [20]:
india_weather_humidity_df = pd.DataFrame(india_weather_humidity)
india_weather_humidity_df

,city,humidity
0,mumbai,82
1,mumbai,85
2,mumbai,80
3,mumbai,88
4,mumbai,90
5,lucknow,60
6,lucknow,65
7,lucknow,70
8,lucknow,55
9,lucknow,62


In [21]:
india_weather_precipitation_df = pd.DataFrame(india_weather_precipitation)
india_weather_precipitation_df

,city,precipitation
0,mumbai,15.2
1,mumbai,12.5
2,mumbai,20.1
3,mumbai,10.0
4,mumbai,18.3
5,lucknow,5.0
6,lucknow,2.2
7,lucknow,0.0
8,lucknow,3.1
9,lucknow,1.5


In [22]:
india_weather_temperature_df = pd.DataFrame(india_weather_temperature)
india_weather_temperature_df

,city,Temperature
0,mumbai,29
1,mumbai,30
2,mumbai,31
3,mumbai,28
4,mumbai,29
5,lucknow,34
6,lucknow,36
7,lucknow,33
8,lucknow,35
9,lucknow,37


### Merge all the three dataFrames into one single dataFrame using merge function

In [23]:
merged_df_1 = india_weather_temperature_df.merge(india_weather_humidity_df,on=["city"]).merge(india_weather_precipitation_df,on=["city"])
merged_df_1

,city,Temperature,humidity,precipitation
0,mumbai,29,82,15.2
1,mumbai,29,82,12.5
2,mumbai,29,82,20.1
3,mumbai,29,82,10.0
4,mumbai,29,82,18.3
...,...,...,...,...
370,delhi,42,47,0.0
371,delhi,42,47,0.5
372,delhi,42,47,0.0
373,delhi,42,47,1.2


### Why the resultant dataFrame has 374 rows when all the original dataFrames has only 14 rows each?
- Each of my DataFrames (Temperature, Precipitation and Humidity) has only 14 rows.
- But the key I am merging on is only city
- That means if a city like ```"Mumbai"``` appears multiple time in ```india_weather_temperature_df``` (say 7 times for different days), and also multiple times in ```india_weather_humidity_df``` (say 7 times) pandas will do a cartesian join i .e ```7 (temperature rows for Mumbai) × 7 (humidity rows for Mumbai) = 49 rows```

### Solution is to create a helper column and index each row of every dataFrames that is to be merged as a single dataFrame. Then use that column along side with the city column to merge the dataFrames into one single dataFrame.

#### Index every dataFrame and save those indexes inside a helper column 
- **NOTICE** : I am dropping this helper column called "row" because I con't want to add any additional column into the dataFrame

In [24]:
india_weather_temperature_df["row"] = india_weather_temperature_df.groupby("city").cumcount()
india_weather_humidity_df["row"] = india_weather_humidity_df.groupby("city").cumcount()
india_weather_precipitation_df["row"] = india_weather_precipitation_df.groupby("city").cumcount()

#### After that you can merge all these three dataFrames into one by utilizing ```city``` and ```row``` columns in them and dropping the row column after the merge process is complete.

In [25]:
merged_df_fixed = india_weather_temperature_df.merge(india_weather_humidity_df,on=["city","row"]).merge(india_weather_precipitation_df,on=["city","row"]).drop(columns="row")
merged_df_fixed

,city,Temperature,humidity,precipitation
0,mumbai,29,82,15.2
1,mumbai,30,85,12.5
2,mumbai,31,80,20.1
3,mumbai,28,88,10.0
4,mumbai,29,90,18.3
5,lucknow,34,60,5.0
6,lucknow,36,65,2.2
7,lucknow,33,70,0.0
8,lucknow,35,55,3.1
9,lucknow,37,62,1.5


## How to handle uncleaned and messy data when merging multiple dataFrames into a single dataFrame.

### Create data in python dictionaries

In [13]:
india_weather_humidity = {
    "city":  ["lucknow"] * 5 + ["mumbai"] * 5 + ["delhi"] * 5,
    "humidity": [
        # Lucknow (moderate humidity)
        60, 65, 70, 55, 62,
        # Mumbai (humid, coastal)
        82, 85, 80, 88, 90,
        # Delhi (dry, lower humidity)
        45, 50, 40, 42, 47
    ]
}

india_weather_precipitation = {
    "city": ["mumbai"] * 5 + ["lucknow"] * 5 + ["delhi"] * 5,
    "precipitation": [
        # Mumbai
        15.2, 12.5, 20.1, 10.0, 18.3, 
        # Lucknow
        5.0, 2.2, 0.0, 3.1, 1.5,    
        # Delhi (with missing values)
        0.0, 0.5, 0.0, None, None       
    ]
}

india_weather_temperature = {
    "city": ["mumbai"] * 5 + ["delhi"] * 5 + ["lucknow"] * 5,
    "Temperature": [
        # Mumbai (warm and stable)
        29, 30, 31, 28, 29,
        # Delhi (very hot summers)
        38, 40, 39, 41, 42,
        # Lucknow (hotter variation)
        34, 36, 33, 35, 37,
    ]
}

### Create dataFrame from the data created earlier

In [14]:
messy_df_india_temperature = pd.DataFrame(india_weather_temperature)
messy_df_india_temperature

,city,Temperature
0,mumbai,29
1,mumbai,30
2,mumbai,31
3,mumbai,28
4,mumbai,29
5,delhi,38
6,delhi,40
7,delhi,39
8,delhi,41
9,delhi,42


In [15]:
messy_df_india_precipitation = pd.DataFrame(india_weather_precipitation)
messy_df_india_precipitation

,city,precipitation
0,mumbai,15.2
1,mumbai,12.5
2,mumbai,20.1
3,mumbai,10.0
4,mumbai,18.3
5,lucknow,5.0
6,lucknow,2.2
7,lucknow,0.0
8,lucknow,3.1
9,lucknow,1.5


In [16]:
messy_df_india_humidity = pd.DataFrame(india_weather_humidity)
messy_df_india_humidity

,city,humidity
0,lucknow,60
1,lucknow,65
2,lucknow,70
3,lucknow,55
4,lucknow,62
5,mumbai,82
6,mumbai,85
7,mumbai,80
8,mumbai,88
9,mumbai,90


### Group all the data in the dataFrames and Index the dataFrames and create a helping column called rows to store the generated indexes in the respective dataFrame

In [17]:
messy_df_india_humidity["row"] = messy_df_india_humidity.groupby("city").cumcount()
messy_df_india_precipitation["row"] = messy_df_india_precipitation.groupby("city").cumcount()
messy_df_india_temperature["row"] = messy_df_india_temperature.groupby("city").cumcount()

In [18]:
merged_df = (messy_df_india_humidity.merge(messy_df_india_precipitation, on=["city","row"]).merge(messy_df_india_temperature,on=["city","row"]).drop(columns="row"))
merged_df

,city,humidity,precipitation,Temperature
0,lucknow,60,5.0,34
1,lucknow,65,2.2,36
2,lucknow,70,0.0,33
3,lucknow,55,3.1,35
4,lucknow,62,1.5,37
5,mumbai,82,15.2,29
6,mumbai,85,12.5,30
7,mumbai,80,20.1,31
8,mumbai,88,10.0,28
9,mumbai,90,18.3,29


### This method works on both the senarios where data is cleaned and when data is not cleaned.